<a href="https://colab.research.google.com/github/K-Musty/100-days_of_code/blob/main/acoustic_leak_detection_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### ACOUSTIC LEAK DETECTION - CORRECTED TRAINING PIPELINE

##### CELL 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Drive mounted")

Mounted at /content/drive
✅ Drive mounted


##### CELL 2: Clone/Pull Your Repository

In [2]:
from google.colab import userdata
# import os

# Retrieve your Personal Access Token from Colab secrets
PAT = userdata.get('PAT')

# Remove old directory if exists (clean start)
!rm -rf acoustic-leak-detection

# os.system(f"git clone https://{PAT}@github.com/K-Musty/acoustic-leak-detection.git")
!git clone https://{PAT}@github.com/K-Musty/acoustic-leak-detection.git
%cd acoustic-leak-detection

# Pull latest changes (just in case)
!git pull

print("✅ Repository ready")


Cloning into 'acoustic-leak-detection'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 157 (delta 22), reused 48 (delta 15), pack-reused 83 (from 1)
Receiving objects: 100% (157/157), 149.56 MiB | 19.99 MiB/s, done.
Resolving deltas: 100% (48/48), done.
Updating files: 100% (72/72), done.
/content/acoustic-leak-detection
Already up to date.
✅ Repository ready


##### CELL 3: Install Dependencies

In [3]:

!pip install torchaudio numpy pandas matplotlib scikit-learn librosa tqdm pyyaml -q

print("✅ Dependencies installed")


✅ Dependencies installed


##### CELL 4: Copy Processed Data from Google Drive

In [4]:
# Only processed data is needed (raw is already converted)
!cp -r /content/drive/MyDrive/acoustic-leak-detection/data/processed ./data/

print("✅ Data copied")


✅ Data copied


##### CELL 5: Verify and Reprocess GPLA (Fix Labels if Needed)

In [5]:
# Reprocess GPLA to ensure labels are 0-indexed (0-11)
!python src/data/preprocess_gpla.py

# Verify labels
!python -c "import numpy as np; y=np.load('data/processed/gpla_v2/y_train.npy'); print('Unique labels:', np.unique(y))"

# Expected output: Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11]


📂 Loading GPLA data...
Traceback (most recent call last):
  File "/content/acoustic-leak-detection/src/data/preprocess_gpla.py", line 71, in <module>
    preprocess_gpla()
  File "/content/acoustic-leak-detection/src/data/preprocess_gpla.py", line 18, in preprocess_gpla
    X = pd.read_excel(f'{raw_dir}/data.xlsx', header=None).values.astype(np.float32)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/excel/_base.py", line 495, in read_excel
    io = ExcelFile(
         ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/excel/_base.py", line 1550, in __init__
    ext = inspect_excel_format(
          ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/excel/_base.py", line 1402, in inspect_excel_format
    with get_handle(
         ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/common.py", line 882, in get_handle
    handle = open(handle, ioargs.mode)

In [9]:
# Verify labels are correctly 0-indexed (0-11)
!python -c "import numpy as np; y=np.load('data/processed/gpla_v2/y_train.npy'); print('Unique labels:', np.unique(y))"

Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11]


##### CELL 6: Ensure No Label Adjustment in train.py

In [10]:
# Remove any line that subtracts 1 from query_y (if present)
!sed -i '/query_y = query_y - 1/d' src/train.py

print("✅ Removed any label adjustment")

✅ Removed any label adjustment


In [14]:
%%writefile src/models/conformer_encoder.py
# src/models/conformer_encoder.py (Fixed)
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvolutionModule(nn.Module):
    """Depthwise convolution module - FIXED."""
    def __init__(self, dim, kernel_size=31, dropout=0.1):
        super().__init__()
        self.pointwise_conv1 = nn.Conv1d(dim, 2*dim, kernel_size=1)
        self.depthwise_conv = nn.Conv1d(
            2*dim, 2*dim, kernel_size=kernel_size,
            padding=kernel_size//2, groups=2*dim
        )
        # FIX: GLU reduces 2*dim → dim, so pointwise_conv2 expects dim input
        self.pointwise_conv2 = nn.Conv1d(dim, dim, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.BatchNorm1d(dim)

    def forward(self, x):
        # x: (batch, seq, dim)
        x = x.transpose(1, 2)          # (batch, dim, seq)
        x = self.pointwise_conv1(x)    # (batch, 2*dim, seq)
        x = self.depthwise_conv(x)     # (batch, 2*dim, seq)
        x = F.glu(x, dim=1)            # (batch, dim, seq) ← GLU halves the channels
        x = self.pointwise_conv2(x)    # (batch, dim, seq) ← now expects dim, not 2*dim
        x = self.dropout(x)
        x = x.transpose(1, 2)          # (batch, seq, dim)
        return x

class ConformerBlock(nn.Module):
    def __init__(self, dim, num_heads, ffn_dim, kernel_size=31, dropout=0.1):
        super().__init__()
        self.ffn1 = nn.Sequential(
            nn.Linear(dim, ffn_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, dim),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(dim)

        self.mha = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)

        self.conv = ConvolutionModule(dim, kernel_size, dropout)
        self.norm3 = nn.LayerNorm(dim)

        self.ffn2 = nn.Sequential(
            nn.Linear(dim, ffn_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, dim),
            nn.Dropout(dropout)
        )
        self.norm4 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: (batch, seq, dim)
        x = x + 0.5 * self.ffn1(x)
        x = self.norm1(x)

        attn_out, _ = self.mha(x, x, x)
        x = x + attn_out
        x = self.norm2(x)

        x = x + self.conv(x)
        x = self.norm3(x)

        x = x + 0.5 * self.ffn2(x)
        x = self.norm4(x)
        return x

class ConformerEncoder(nn.Module):
    def __init__(self,
                 input_dim=2000,
                 output_dim=128,
                 num_heads=4,
                 ffn_dim=256,
                 num_layers=4,
                 kernel_size=31,
                 dropout=0.1):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, ffn_dim)

        self.blocks = nn.ModuleList([
            ConformerBlock(ffn_dim, num_heads, ffn_dim, kernel_size, dropout)
            for _ in range(num_layers)
        ])

        self.output_proj = nn.Linear(ffn_dim, output_dim)
        self.norm_out = nn.LayerNorm(output_dim)

    def forward(self, x):
        # x: (batch, input_dim)
        x = self.input_proj(x)
        x = x.unsqueeze(1)

        for block in self.blocks:
            x = block(x)

        x = x.squeeze(1)
        x = self.output_proj(x)
        x = self.norm_out(x)
        return x

Overwriting src/models/conformer_encoder.py


In [ ]:
print("✅ Conformer encoder fixed")

##### CELL 7: Quick Test – Baseline (5 Epochs) to Verify Setup

In [ ]:
!PYTHONPATH=. python src/train.py configs/baseline.yaml

# If you see "Device: cuda" and loss decreases from ~2.48, it's working.
# If you see "Device: cpu", set runtime to GPU and restart.

🔧 Device: cuda
✅ Loaded GPLA (train): 468 samples, 12 classes
   Classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] (0-indexed)
✅ Loaded GPLA (val): 156 samples, 12 classes
   Classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] (0-indexed)
📊 Model parameters: 3,231,360

🚀 Starting 100 epochs...
Epoch   1/100 | Loss: 0.7952 | Val Acc: 58.87%
Epoch   5/100 | Loss: 0.0033 | Val Acc: 60.97%
Epoch  10/100 | Loss: 0.0011 | Val Acc: 62.50%
Epoch  15/100 | Loss: 0.0006 | Val Acc: 62.63%
Epoch  20/100 | Loss: 0.0004 | Val Acc: 64.43%
Epoch  25/100 | Loss: 0.0003 | Val Acc: 62.37%
Epoch  30/100 | Loss: 0.0007 | Val Acc: 64.23%


##### CELL 8: Run Full Baseline (50 Epochs)

In [ ]:
!PYTHONPATH=. python src/train.py configs/baseline.yaml

##### CELL 9: Run Full Attention (50 Epochs)

In [ ]:
!PYTHONPATH=. python src/train.py configs/attention.yaml

##### CELL 10: Save Results to Google Drive with Timestamp

In [ ]:
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
!mkdir -p /content/drive/MyDrive/acoustic-leak-detection/experiments_$timestamp
!cp -r experiments/* /content/drive/MyDrive/acoustic-leak-detection/experiments_$timestamp/

print(f"✅ Results saved to experiments_{timestamp}")

##### CELL 11: (Optional) List Saved Checkpoints and Show Summary of Results

In [ ]:
!ls -la /content/drive/MyDrive/acoustic-leak-detection/experiments_*/best_*.pt

print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)

# Check if baseline results exist
!ls -la experiments/best_gpla_baseline.pt 2>/dev/null && echo "✅ Baseline: trained" || echo "❌ Baseline: not found"

# Check if attention results exist
!ls -la experiments/best_gpla_attention.pt 2>/dev/null && echo "✅ Attention: trained" || echo "❌ Attention: not found"

print("\n📊 Check Drive for full results.")

##### CELL 12: (Optional) Download Results to Local Machine

In [ ]:

# Uncomment and run to download a zip of results to your local machine
# !zip -r experiments_results.zip experiments/
# from google.colab import files
# files.download('experiments_results.zip')


# ============================================================
# END OF NOTEBOOK
# ============================================================
print("\n✅ All done! Check your Google Drive for results.")